# 📉 Cuaderno 01 · Derivación numérica y diferenciación en tiempo real

Un encoder, una cámara, un termopar: todos entregan **muestras**, no funciones
matemáticas derivables analíticamente. En este cuaderno vas a estimar
$\frac{\mathrm{d}y}{\mathrm{d}t}$ a partir de datos y a decidir **qué
diferencia usar** según el ruido, el lazo de control y dónde está la muestra.

> Las preguntas valen 20 puntos cada una y el caso vale 40. Las actividades a
> mano no restan: son evidencia de que seguiste el procedimiento.


In [ ]:
#@title ⚙️ 1.1 · Entorno — ejecuta esta celda primero
# ---------------------------------------------------------------------------
# No hay nada que editar aquí: la celda prepara el material del curso en tu
# Drive y deja todo listo para trabajar. En Google Colab te va a pedir permiso
# para acceder a tu Drive (es el único permiso que pide esta celda).
# ---------------------------------------------------------------------------
import base64
import io
import os
import sys
import tempfile
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

CUADERNO = "u5_01_derivacion"
CARPETA_MATERIAL = "CursoMN/material"
ORIGEN = "https://raw.githubusercontent.com/ITH-MGL-MN/U5_derivadas_integrales_EDO/main"


try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False


def _raiz_local():
    """Sube desde la carpeta del cuaderno hasta encontrar lib/robomat.py."""
    candidatas = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatas.append(os.path.dirname(os.path.abspath(nb)))
    candidatas.append(os.getcwd())
    for base in candidatas:
        carpeta = Path(base)
        for _ in range(7):
            if (carpeta / "lib" / "robomat.py").exists():
                return carpeta
            carpeta = carpeta.parent
    return None


def _carpeta_material():
    """Deja el material del curso en el Drive del alumno y devuelve su ruta.

    Monta el Drive, actualiza los archivos que el cuaderno necesita y no
    pide nada por teclado.

    Returns:
        Path: carpeta con `lib/` y `grader/`.

    Raises:
        RuntimeError: si falta algún archivo del material (por ejemplo, porque
            todavía no se publicó el paquete del motor).
    """
    from google.colab import drive

    drive.mount("/content/drive")
    base = Path("/content/drive/MyDrive") / CARPETA_MATERIAL
    for relativa in ("lib/robomat.py", "lib/graficos.py", "lib/pantalla.py",
                     "grader/" + CUADERNO + ".txt"):
        destino = base / relativa
        destino.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(f"{ORIGEN}/{relativa}", timeout=60) as respuesta:
                destino.write_bytes(respuesta.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"Falta {relativa} del material del curso (HTTP {exc.code}). "
                "Avisa al profesor."
            ) from exc
    return base


if EN_COLAB:
    RAIZ = _carpeta_material()
else:
    RAIZ = _raiz_local()

if RAIZ is None:
    raise RuntimeError("No encuentro el material del curso: abre el cuaderno desde la carpeta notebooks/.")


def _prepara_motor(raiz, cuaderno):
    """Deja el motor importable y dice en qué modo quedó.

    Con la rama test se usa el código legible (F12 y pdb funcionan). Sin él, se
    descomprime el paquete de `grader/`, que es lo que recibe el alumno.
    """
    if (raiz / "privado").is_dir():
        sys.path.insert(0, str(raiz))
        return "local"
    carpeta = Path(tempfile.mkdtemp(prefix="u5motor_"))
    archivo = raiz / "grader" / (cuaderno + ".txt")
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(archivo.read_bytes()))) as paquete:
        paquete.extractall(carpeta)
    sys.path.insert(0, str(carpeta))
    return "blob"


MODO = _prepara_motor(RAIZ, CUADERNO)
sys.path.insert(0, str(RAIZ / "lib"))

from privado.api import *          # noqa: F403,E402
from graficos import laboratorio   # noqa: E402

# El cuaderno le dice al motor cuál es su cuaderno. En el paquete de Colab ya
# viene fijo; en local hay que declararlo.
import privado.ajustes as _ajustes  # noqa: E402

_ajustes.CUADERNO = CUADERNO
try:
    _ajustes.TAREA_ID = _ajustes.leer_config().get("cuadernos", {}).get(CUADERNO, "")
except FileNotFoundError:
    _ajustes.TAREA_ID = ""

exec((RAIZ / "lib" / "robomat.py").read_text(encoding="utf-8"))   # noqa: S102

print(f"✅ Cuaderno {CUADERNO} listo. Continúa con la celda 1.2.")


In [ ]:
#@title 🪪 1.2 · Cuenta — tus datos
# Tu cuenta fija el número de control y, con él, todos los datos de la unidad:
# el ejercicio a mano, el caso y la versión de cada pregunta.
# Usa tu correo INSTITUCIONAL (el que trae tu número de control): con otra
# cuenta no se puede registrar la calificación.
# En Google Colab se te va a pedir autorizar tu cuenta de Google: es un permiso
# aparte del que pediste para el Drive. En local no se pide nada.
MI_CUENTA = identificacion()


## 2.1 · Qué es derivar numéricamente y por qué duele

La derivada es un límite; con datos nunca llegamos al límite, solo a un
cociente de diferencias. La pregunta del tema no es "cuál es la fórmula" sino
**de qué lado del punto tomo la muestra** y **cuánto separo una muestra de otra**.

Las tres respuestas clásicas son las de las figuras 23.1 a 23.3 del libro:
hacia adelante, hacia atrás y centrada. Cambian dos cosas: el **orden del
error** y la **causalidad** (si para calcular la derivada de hoy necesito una
muestra de mañana, no puedo usarla en un lazo de control).

### Figura 23.4 · el caso que sí existe en la práctica

![Figura 23.4](https://raw.githubusercontent.com/ITH-MGL-MN/U5_derivadas_integrales_EDO/main/docs/figuras/libro/fig23_4_temperatura.png)

Un termómetro se usa para medir la temperatura del suelo a 0, 1.25 y 3.75 cm de
profundidad y obtiene 13.5, 12 y 10 °C. Para estimar
$\frac{\mathrm{d}T}{\mathrm{d}z}$ hay que dividir diferencias de temperatura
entre diferencias de profundidad **que no son iguales**. Ese es el caso normal:
los instrumentos no muestrean parejo.

*Qué muestra*: tres pares (profundidad, temperatura) sobre un eje vertical que
crece hacia abajo.

*Qué concluye*: la temperatura baja al profundizar, y con tres puntos desiguales
la diferencia centrada se generaliza dividiendo entre el intervalo total.

*Qué limitación tiene*: con tres puntos no hay forma de estimar el error sin
suponer algo sobre la curvatura.


### Figura 23.5 · por qué el ruido es el enemigo

Las dos imágenes siguientes son las dos mitades de la figura 23.5 del libro. En
la primera, los datos son limpios y la derivada sale suave; en la segunda, a los
mismos datos se les cambió un valor **muy poco** y la derivada se volvió un
zigzag.

![Figura 23.5, paneles a y c](https://raw.githubusercontent.com/ITH-MGL-MN/U5_derivadas_integrales_EDO/main/docs/figuras/libro/fig23_5_ruido_1.png)

![Figura 23.5, paneles b y d](https://raw.githubusercontent.com/ITH-MGL-MN/U5_derivadas_integrales_EDO/main/docs/figuras/libro/fig23_5_ruido_2.png)

*Qué muestra*: arriba los datos, abajo su derivada, en los dos casos.

*Qué concluye*: un error pequeño en $y$ se convierte en un error grande en
$\frac{\mathrm{d}y}{\mathrm{d}x}$.

*Qué limitación tiene*: el método de diferencias no distingue entre una
variación real y un error de medición; por eso antes de derivar hay que decidir
si se filtra.

| Si necesitas… | Usa | Porque |
|---|---|---|
| Derivar dentro de un controlador | diferencia hacia atrás | Es la única que solo usa el pasado |
| Derivar con la mejor precisión y sin prisa | diferencia centrada | Su error es $O(h^2)$ y cuesta lo mismo |
| Derivar una señal ruidosa | filtrar y después centrada | El ruido se amplifica como $1/h$ |

Todo el catálogo (con los nombres exactos en Python y en MATLAB, y las figuras
del libro) está en la [guía de derivación numérica](https://github.com/ITH-MGL-MN/U5_derivadas_integrales_EDO/blob/main/docs/guias/catalogo_01_derivacion.md).


In [ ]:
#@title 🧪 2.2 · Pregunta 1
pregunta(1)


## 3.1 · Diferencias finitas: las tres fórmulas y su error

Con una malla de paso $h$, el libro escribe las tres primeras derivadas así:

$$
f'(x_i) \approx \frac{f(x_{i+1}) - f(x_i)}{h}
\qquad\text{adelante, error } O(h)
$$

$$
f'(x_i) \approx \frac{f(x_i) - f(x_{i-1})}{h}
\qquad\text{atrás, error } O(h)
$$

$$
f'(x_i) \approx \frac{f(x_{i+1}) - f(x_{i-1})}{2h}
\qquad\text{centrada, error } O(h^2)
$$

La centrada gana un orden **gratis**: usa los mismos dos valores, solo que uno a
cada lado. El precio es que hay que esperar la muestra siguiente, y que en el
primer y el último punto de la señal no se puede aplicar.

### La diferencia hacia atrás, que es la que corre en un microcontrolador

Es la única **causal**: para estimar la derivada en la muestra $i$ usa $i$ e
$i-1$, es decir, solo lo que ya ocurrió. Cuesta una resta y una división, y por
eso es la que aparece en un lazo de control a alta velocidad.

Su forma con espaciado desigual —que es la que necesitas cuando el
microcontrolador no muestrea parejo— es la misma división entre el intervalo
real:

$$
f'(x_i) \approx \frac{y_i - y_{i-1}}{t_i - t_{i-1}}
$$

**Lo que se te pide**: escribir dos funciones, la de atrás y la centrada. Reciben
la tabla completa y el índice, y devuelven un número.


In [ ]:
#@title ✍️ 3.2 · Tu ejercicio a mano
MANO = mano_enunciado("derivada_atras")


In [ ]:
#@title ✍️ 3.2 · Comprueba tu resultado
# Escribe aquí el número que calculaste (con 4 decimales).
ACIERTO = mano_comprueba("derivada_atras", 0.0)


### 3.3 · Programa tus dos diferencias

Completa las funciones de la celda siguiente **sin cambiar sus nombres ni sus
parámetros**. Después ejecútalas con `practica(...)`, que las compara con la
referencia en tres casos cada una: el ejemplo 23.1 del libro, un caso con
espaciado desigual y tu propio caso sembrado con tu número de control.

- **Entra**: `t` (tiempos), `y` (valores) e `i` (índice de la muestra).
- **Sale**: un `float`, la derivada estimada.
- **Casos raros**: si `i` no tiene la muestra que hace falta, la función debe
  avisar con `ValueError` en vez de devolver un número inventado.

<details><summary>Pista 1 · dónde están en el libro</summary>

Son las fórmulas de las figuras 23.2 y 23.3: "diferencias divididas hacia atrás"
y "centradas".

</details>

<details><summary>Pista 2 · qué forma tienen las respuestas</summary>

Las dos son dos restas y una división. En la de atrás restas los valores de
`i` e `i-1`; en la centrada, los de `i+1` e `i-1`. El denominador no es `h`:
es la diferencia de tiempos, que funciona igual con espaciado uniforme o no.

</details>

<details><summary>Pista 3 · qué propiedad del problema mirar</summary>

Fíjate en el signo: si la señal sube, la derivada debe salir positiva. Y fíjate
en el rango de `i` que cada una puede aceptar.

</details>


In [ ]:
#@title 💻 3.3 · Programa las dos funciones
def derivada_atras(t, y, i):
    """Derivada en la muestra i por diferencia hacia atrás.

    Args:
        t: tiempos de las muestras, crecientes.
        y: valores muestreados, misma longitud que t.
        i: índice de la muestra donde se estima la derivada (1 o mayor).

    Returns:
        float: la derivada estimada.

    Raises:
        ValueError: si i no tiene muestra anterior.
    """
    raise NotImplementedError("Escribe aquí tu diferencia hacia atrás")


def derivada_centrada(t, y, i):
    """Derivada en la muestra i por diferencia centrada.

    Args:
        t: tiempos de las muestras, crecientes.
        y: valores muestreados, misma longitud que t.
        i: índice de la muestra donde se estima la derivada.

    Returns:
        float: la derivada estimada.

    Raises:
        ValueError: si i no tiene una muestra a cada lado.
    """
    raise NotImplementedError("Escribe aquí tu diferencia centrada")


In [ ]:
#@title ▶️ 3.3 · Compáralas con la referencia
APROBADA_ATRAS = practica(derivada_atras)


In [ ]:
#@title ▶️ 3.3 · La centrada, contra la referencia
APROBADA_CENTRADA = practica(derivada_centrada)


In [ ]:
#@title 📊 3.4 · Laboratorio
laboratorio("derivada_atras")


### 3.4 · Qué observar en el laboratorio

El laboratorio dibuja tres cosas. Míralas en este orden:

1. **La secante.** Es la derivada: la pendiente de la recta que une dos
   muestras. Si la señal es curva, la secante de un lado se queda corta o se
   pasa; la centrada promedia dos errores de signo contrario.
2. **La estimación contra la exacta.** Con la malla gruesa del ejemplo (h = 0.1)
   la diferencia hacia atrás se separa de la curva exacta sobre todo donde la
   curvatura es grande. La centrada casi se pega.
3. **El error al refinar.** Es la gráfica que decide el tema. En escala
   logarítmica, una recta de pendiente 1 significa error $O(h)$ y una de
   pendiente 2 significa $O(h^2)$. Al duplicar los intervalos, la primera baja a
   la mitad y la segunda a la cuarta parte.

> **Ojo con `np.gradient`.** Hace lo mismo que la centrada, pero además resuelve
> los extremos con diferencias de un solo lado. Por eso en el laboratorio los
> dos arreglos coinciden en el interior y solo difieren en los bordes.


In [ ]:
#@title 🧪 3.5 · Pregunta 2
pregunta(2)


## 4.1 · Lo que no se programa, pero sí se elige

Ya tienes la diferencia hacia atrás. Los demás casos del capítulo 23 no se
escriben a mano: se configuran. Lo que hay que saber es **cuál pedir**.

- **Segunda derivada.** Se obtiene derivando dos veces, y ahí el ruido se vuelve
  el problema principal: si al derivar una vez el error crece como $1/h$, al
  derivar dos veces crece como $1/h^2$. Nunca se deriva dos veces una señal
  ruidosa sin filtrar.
- **Espaciado desigual.** La diferencia centrada se generaliza dividiendo entre
  el intervalo total entre las dos muestras que rodean al punto. Es la ecuación
  (23.9) del libro aplicada al caso lineal.
- **Extrapolación de Richardson.** Combina dos estimaciones con pasos distintos
  para cancelar el término de error dominante (ecuación 23.8). Gana orden, pero
  amplifica el ruido: solo con datos limpios.
- **Filtrado.** Una media móvil de $N$ muestras baja el ruido y también suaviza
  la señal real. Existe un $N$ que equilibra las dos cosas, y no es el mismo
  para todas las señales.

## 4.2 · Cómo se llaman en Python y en MATLAB

| Necesitas | Python | MATLAB |
|---|---|---|
| Diferencias simples | `np.diff(y)` | `diff(y)` |
| Derivada centrada, con ejes | `np.gradient(y, t)` | `gradient(y, t)` |
| Media móvil | `np.convolve(y, np.ones(N)/N, "same")` | `filter(ones(1,N)/N, 1, y)` |
| Suavizado que respeta la forma | `scipy.signal.savgol_filter` | `sgolayfilt` |
| Segunda derivada | `np.gradient(np.gradient(y, t), t)` | `gradient(gradient(y, t), t)` |

**Cuándo sí y cuándo no.** `np.gradient` es la opción por defecto cuando ya
tienes toda la señal en memoria: resuelve los bordes y acepta tiempos
irregulares. `np.diff` **no** divide entre `dt` ni resuelve bordes: sirve para
ver la diferencia bruta, no para reportar una velocidad. La media móvil es
barata y predecible, pero retrasa la señal media ventana; `savgol_filter`
conserva mejor los picos y cuesta más.


In [ ]:
#@title 🔧 4.3 · Configúralo tú (no lo programes)
# Cambia VENTANA y el RUÍDO del sensor y vuelve a ejecutar. El laboratorio
# dibuja la señal, su derivada y el error según el ancho de la ventana.
VENTANA = 7          # muestras que promedia la media móvil
RUIDO = 0.01         # desviación estándar del sensor, en unidades de la señal

FILTRO = laboratorio("filtrado", {"ventana": VENTANA, "ruido": RUIDO})


### 4.4 · Qué observar en el laboratorio del filtro

Los tres paneles cuentan la misma historia en tres pasos:

1. **La señal.** La línea gris es lo que entrega el sensor: la curva real más
   ruido. La azul es la misma señal después de promediar `VENTANA` muestras. A
   simple vista se parecen; la diferencia está en la derivada.
2. **La derivada.** Al derivar la señal sin filtrar (gris), el ruido se
   multiplica y tapa la curva real. La derivada de la señal filtrada (azul) sí
   sigue a la exacta (negro a rayas). Ese es el motivo de filtrar antes de
   derivar.
3. **El ancho de la ventana.** El error no baja siempre: con ventanas pequeñas
   manda el ruido y con ventanas grandes el filtro se come la señal y la
   derivada se aplana. Hay un mínimo, y está marcado. Si eliges a ojo, casi
   siempre eliges de más: promediar 30 muestras se ve muy limpio y es un error
   grande.

Cambia `VENTANA` y vuelve a ejecutar la celda: la línea azul del panel 3 se
mueve. Con `VENTANA = 1` no hay filtro y el resultado es el de la señal cruda.


## 5.1 · Caso de ingeniería: aceleración de un brazo robótico

Un brazo gira con un movimiento suave, pero el encoder que mide el ángulo
entrega ruido. Con esas muestras hay que reportar **la velocidad angular
máxima**, y para eso hay que decidir tres cosas: cada cuánto se muestrea, si se
filtra, y qué diferencia se usa.

Ejecuta la celda siguiente: te va a dar **tus** datos (amplitud, frecuencia,
nivel de ruido y el arreglo de muestras) y la lista de métodos admitidos con lo
que vale cada uno. Elige tú.

<details><summary>Pista 1 · por dónde empezar</summary>

Con ruido, derivar directo es lo peor que puedes hacer: la figura 23.5 lo
muestra con datos de juguete, y tus datos son iguales. Antes de derivar, suaviza.

</details>

<details><summary>Pista 2 · qué forma tiene la respuesta</summary>

Un solo número, en rad/s, dentro de un diccionario con la clave
`"velocidad_max"`. Los extremos de la señal no se usan: la centrada no se puede
aplicar ahí.

</details>

<details><summary>Pista 3 · qué propiedad mirar</summary>

`np.convolve(..., mode="same")` mantiene el mismo tamaño que la señal, así que
el índice de la muestra i sigue significando lo mismo antes y después de
filtrar. La velocidad máxima es el máximo del **valor absoluto**: el brazo gira
en los dos sentidos.

</details>


In [ ]:
#@title 🏭 5.1 · Tus datos y los métodos admitidos
CASO = caso()


### 5.1.1 · Tu solución

Completa los tres huecos. Nada de esto es de otra materia: la media móvil, el
máximo y el valor absoluto ya están resueltos; lo que falta es **tu** decisión.


In [ ]:
#@title 🛠️ 5.1.1 · Solución del caso
import numpy as np

VENTANA = 7                      # muestras de la media móvil
DATOS = CASO["datos"]
t = DATOS["t"]
theta = DATOS["theta"]

# Paso 1: quita el ruido sin cambiar el tamaño de la señal.
suave = ____

# Paso 2: estima la velocidad angular con tu función del numeral 3.3.
#         Recorre las muestras interiores (la centrada no se puede en los bordes).
velocidad = [____ for i in range(1, len(t) - 1)]

# Paso 3: el brazo gira en los dos sentidos, así que interesa el valor absoluto.
velocidad_max = ____

print(f"Velocidad angular máxima: {velocidad_max:.4f} rad/s")


In [ ]:
#@title ✅ 5.1.2 · Revisar el caso
# Declara el método que usaste y entrega tu número.
VEREDICTO = caso_revisar(metodo="derivada_centrada", resultados={"velocidad_max": velocidad_max})


In [ ]:
#@title 🧪 5.1.3 · Pregunta 3
pregunta(3)


## 6 · Cierre

`calificar()` suma tus tres preguntas y el caso, y normaliza a 100. El resultado
queda en `NOTA` y `TOPE`, por si quieres consultarlo después.

`enviar()` manda el resultado al backend del curso y `consultar_calificacion()`
te dice lo que quedó guardado. Sin conexión, `enviar()` avisa y no falla.

Si te quedó duda de algún método que no programaste, la
[guía de métodos](https://github.com/ITH-MGL-MN/U5_derivadas_integrales_EDO/blob/main/docs/guias/README.md) tiene la tabla de decisión rápida y los
nombres en Python y en MATLAB.


In [ ]:
#@title 🧮 6.1 · Calificar
NOTA, TOPE = calificar()


In [ ]:
#@title 📤 6.2 · Enviar
ENVIO = enviar()


In [ ]:
#@title 🔎 6.3 · Consultar la calificación guardada
GUARDADO = consultar_calificacion()
